In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms.v2 as tfs

import os
import json
from PIL import Image

from tqdm import tqdm

### Класс для загрузки датасета

In [30]:
class DigitDataset(data.Dataset):
    def __init__(self, path, train=True, transform=None):
        self.path = os.path.join(path, "train" if train else "test")
        self.transform = transform

        with open(os.path.join(path, "format.json"), "r") as file:
            self.format = json.load(file)
        
        self.files = []
        self.length = 0
        self.targets = torch.eye(10)

        for dir_, target_ in self.format.items():
            cl_path = os.path.join(self.path, dir_)
            list_filenames = os.listdir(cl_path)
            self.length += len(list_filenames)
            self.files.extend(map(lambda x_: (os.path.join(cl_path, x_), target_), list_filenames))

    def __getitem__(self, item):
        img_path, target = self.files[item]
        img_pil = Image.open(img_path)

        if self.transform:
            img_tensor = self.transform(img_pil).ravel().float() / 255.
        
        return img_tensor, self.targets[target]

    def __len__(self):
        return self.length

In [39]:
to_tensor = tfs.ToImage()
cur_dir = os.getcwd().split('/')[-2]
d_train = DigitDataset(path=f"/Users/sidorovegor/Desktop/projects/python/dl_selfedu_course/datasets/dataset_MNIST", train=True, transform=to_tensor)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)

### Проверка работы написанного класса

In [44]:
it = iter(train_data)
x, y = next(it)
x.shape, y.shape

(torch.Size([32, 784]), torch.Size([32, 10]))

### Класс с моделью для классификации

In [56]:
class DigitNN(nn.Module):
    def __init__(self, inp, hid, out):
        super().__init__()
        self.l1 = nn.Linear(inp, hid)
        self.l2 = nn.Linear(hid, out)

    def forward(self, x):
        x = self.l1(x).relu()
        x = self.l2(x)

        return x

In [57]:
model = DigitNN(28 * 28, 32, 10)
model.train()

DigitNN(
  (l1): Linear(in_features=784, out_features=32, bias=True)
  (l2): Linear(in_features=32, out_features=10, bias=True)
)

In [58]:
optimizer = optim.Adam(params=model.parameters(), lr=0.01)
loss_func = nn.CrossEntropyLoss()

In [59]:
epochs = 2
for _e in range(epochs):
    train_bar = tqdm(train_data, leave=True)

    loss_mean = 0
    lm_count = 0

    for x_train, y_train in train_bar:
        pred = model(x_train)
        loss = loss_func(pred, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1 / lm_count * loss.item() + (1 - 1 / lm_count) * loss_mean

        train_bar.set_description(f"Epoch {_e + 1}/{epochs}], loss_mean={loss_mean:.3f}:")

Epoch 2/2], loss_mean=0.188:: 100%|██████████| 1875/1875 [00:10<00:00, 176.12it/s]


In [62]:
d_test = DigitDataset(path=f"/Users/sidorovegor/Desktop/projects/python/dl_selfedu_course/datasets/dataset_MNIST", train=False, transform=to_tensor)
test_data = data.DataLoader(d_test, batch_size=500, shuffle=False)

In [63]:
Q = 0
model.eval()
for x_test, y_test in test_data:
    with torch.no_grad():
        pred = model(x_test)
        pred = torch.argmax(pred, dim=1)
        y = torch.argmax(y_test, dim=1)

        Q += (pred == y).sum().item()

Q /= len(d_test)
Q

0.9477